# COMP9444 26T2 Project 005: ResNet50 and VGG16 Baselines

This notebook keeps the ResNet50-FCN and VGG16-FCN baseline implementations in the main `9444` project folder. It trains one selected baseline at a time and selects the best epoch by validation mIoU.

Disk output is intentionally limited to one checkpoint file:
`Best_Model/<MODEL_NAME>_best_model.pt`.


## 1. Setup and Imports


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, VGG16_Weights, resnet50, vgg16
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print(
    "Available device:",
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu",
)


---
## 2. Paths and Experiment Configuration

`MODEL_NAME` controls which baseline is trained. Accepted values are `"ResNet50"` and `"VGG16"`; the notebook does not train or compare both models in one run.


In [ ]:
ROOT = Path.cwd().resolve()
DATA_ROOT = ROOT / "UECFOODPIX" / "data"
SPLIT_FILE = ROOT / "uecfoodpix_split_train3000_val500.json"
BEST_MODEL_DIR = ROOT / "Best_Model"

MODEL_NAME = "VGG16"  # "VGG16" or "ResNet50"
NUM_CLASSES = 103
WIDTH = 384
HEIGHT = 320
SEED = 9444


MODEL_CONFIGS = {
    "VGG16": {
        "batch_size": 2,
        "max_epochs": 20,
        "learning_rate": 1e-4,
        "weight_decay": 1e-3,
        "dropout": 0.5,
        "early_stopping_patience": 3,
        "pretrained": True,
    },
    "ResNet50": {
        "batch_size": 2,
        "max_epochs": 25,
        "learning_rate": 2e-4,
        "weight_decay": 5e-4,
        "dropout": 0.3,
        "early_stopping_patience": 4,
        "pretrained": True,
    },
}

def get_model_config(model_name):
    if model_name not in MODEL_CONFIGS:
        raise ValueError(f"MODEL_NAME must be one of {sorted(MODEL_CONFIGS)}")
    return dict(MODEL_CONFIGS[model_name])


CONFIG = get_model_config(MODEL_NAME)

BATCH_SIZE = CONFIG["batch_size"]
MAX_EPOCHS = CONFIG["max_epochs"]
LEARNING_RATE = CONFIG["learning_rate"]
WEIGHT_DECAY = CONFIG["weight_decay"]
DROPOUT = CONFIG["dropout"]
EARLY_STOPPING_PATIENCE = CONFIG["early_stopping_patience"]
PRETRAINED = CONFIG["pretrained"]
NUM_WORKERS = 0

assert DATA_ROOT.exists(), f"Missing dataset folder: {DATA_ROOT}"
assert SPLIT_FILE.exists(), f"Missing split file: {SPLIT_FILE}"
assert MODEL_NAME in MODEL_CONFIGS

BEST_MODEL_PATH = BEST_MODEL_DIR / f"{MODEL_NAME}_best_model.pt"
print(f"Project root: {ROOT}")
print(f"Dataset root: {DATA_ROOT}")
print(f"Selected model: {MODEL_NAME}")
print(f"Active config: {CONFIG}")
print(f"Best checkpoint will be saved to: {BEST_MODEL_PATH}")


---
## 3. Fixed Training and Validation Split

The same 3,000/500 JSON split is used for both baselines, but only the selected model is trained in the current run. The official test set remains separate and is not used for checkpoint selection.


In [ ]:
def load_split(split_file):
    split_data = json.loads(Path(split_file).read_text(encoding="utf-8"))
    train_ids = [str(item) for item in split_data["train_ids"]]
    val_ids = [str(item) for item in split_data["val_ids"]]

    assert len(train_ids) == split_data["train_size"]
    assert len(val_ids) == split_data["val_size"]
    assert len(train_ids) == len(set(train_ids))
    assert len(val_ids) == len(set(val_ids))
    assert not (set(train_ids) & set(val_ids))
    return train_ids, val_ids, split_data


train_ids, val_ids, split_info = load_split(SPLIT_FILE)
print("Training images:", len(train_ids))
print("Validation images:", len(val_ids))
print("Train/validation overlap:", len(set(train_ids) & set(val_ids)))
print("Split seeds:", split_info["train_seed"], split_info["val_seed"])


---
## 4. Dataset and Preprocessing

Images are resized to `384 x 320`, normalized with ImageNet statistics, and paired with nearest-neighbour resized segmentation masks. Training samples use a synchronized random horizontal flip.


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class FoodSegmentationDataset(Dataset):
    def __init__(self, root, ids, split, augment, width, height):
        self.root = Path(root)
        self.ids = list(ids)
        self.split = split
        self.augment = augment
        self.width = width
        self.height = height
        self.image_dir = self.root / "UECFoodPIX" / split / "img"
        self.mask_dir = self.root / "UECFoodPIX" / split / "mask"

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        sample_id = self.ids[index]
        image_path = self.image_dir / f"{sample_id}.jpg"
        mask_path = self.mask_dir / f"{sample_id}.png"

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)

        if self.augment and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

        image = image.resize((self.width, self.height), Image.Resampling.BILINEAR)
        mask = mask.resize((self.width, self.height), Image.Resampling.NEAREST)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(image_array.copy()).permute(2, 0, 1)
        image_tensor = (image_tensor - IMAGENET_MEAN) / IMAGENET_STD

        mask_array = np.asarray(mask)
        if mask_array.ndim == 3:
            mask_array = mask_array[..., 0]
        mask_tensor = torch.from_numpy(mask_array.astype(np.int64, copy=True)).long()
        return image_tensor, mask_tensor


train_dataset = FoodSegmentationDataset(
    DATA_ROOT, train_ids, "train", True, WIDTH, HEIGHT
)
val_dataset = FoodSegmentationDataset(
    DATA_ROOT, val_ids, "train", False, WIDTH, HEIGHT
)

x_sample, y_sample = val_dataset[0]
print("Image tensor:", tuple(x_sample.shape))
print("Mask tensor:", tuple(y_sample.shape))
print("Mask label range:", int(y_sample.min()), "to", int(y_sample.max()))


---
## 5. Model Architectures

Both baselines use a standard torchvision encoder with a lightweight FCN-style segmentation head and bilinear upsampling back to the input resolution.


In [ ]:
class ResNet50FCN(nn.Module):
    def __init__(self, num_classes, dropout=0.2, pretrained=False):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights)
        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(512, num_classes, kernel_size=1),
        )

    def forward(self, x):
        output_size = x.shape[-2:]
        logits = self.classifier(self.encoder(x))
        return F.interpolate(
            logits,
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )


class VGG16FCN(nn.Module):
    def __init__(self, num_classes, dropout=0.2, pretrained=False):
        super().__init__()
        weights = VGG16_Weights.DEFAULT if pretrained else None
        self.encoder = vgg16(weights=weights).features
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(256, num_classes, kernel_size=1),
        )

    def forward(self, x):
        output_size = x.shape[-2:]
        logits = self.classifier(self.encoder(x))
        return F.interpolate(
            logits,
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )


def build_model(model_name, config=None):
    config = get_model_config(model_name) if config is None else config
    dropout = config["dropout"]
    pretrained = config["pretrained"]
    if model_name == "ResNet50":
        return ResNet50FCN(NUM_CLASSES, dropout, pretrained)
    if model_name == "VGG16":
        return VGG16FCN(NUM_CLASSES, dropout, pretrained)
    raise ValueError(f"Unknown model: {model_name}")


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


model_preview = build_model(MODEL_NAME, CONFIG)
print(f"{MODEL_NAME} trainable parameters: {count_parameters(model_preview):,}")
del model_preview


---
## 6. Metrics and Training Utilities

Validation mIoU is the only checkpoint-selection metric. No training curves, CSV files, JSON files, comparison plots, or prediction images are saved.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def update_confusion(confusion, prediction, target):
    prediction = prediction.detach().reshape(-1).cpu().to(torch.int64)
    target = target.detach().reshape(-1).cpu().to(torch.int64)
    valid = (target >= 0) & (target < NUM_CLASSES)
    encoded = target[valid] * NUM_CLASSES + prediction[valid].clamp(0, NUM_CLASSES - 1)
    confusion += torch.bincount(
        encoded,
        minlength=NUM_CLASSES ** 2,
    ).reshape(NUM_CLASSES, NUM_CLASSES)


def calculate_scores(confusion):
    cm = confusion.double()
    correct = cm.diag()
    total = cm.sum().clamp_min(1)
    pixel_accuracy = float(correct.sum() / total)
    union = cm.sum(0) + cm.sum(1) - correct
    valid_iou = union > 0
    mean_iou = float((correct[valid_iou] / union[valid_iou]).mean())
    return pixel_accuracy, mean_iou


def run_epoch(model, loader, loss_fn, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for images, masks in tqdm(
            loader,
            desc="train" if training else "validation",
            leave=False,
        ):
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            loss = loss_fn(logits, masks)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.shape[0]
            update_confusion(confusion, logits.argmax(1), masks)

    pixel_accuracy, mean_iou = calculate_scores(confusion)
    return total_loss / len(loader.dataset), pixel_accuracy, mean_iou, confusion


---
## 7. Train the Selected Baseline

The best validation checkpoint is overwritten only when validation mIoU improves. The saved file name is fixed by the selected model, for example `Best_Model/VGG16_best_model.pt`.


In [ ]:
def train_selected_model(model_name):
    config = get_model_config(model_name)
    best_model_path = BEST_MODEL_DIR / f"{model_name}_best_model.pt"
    seed_everything(SEED)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
    model = build_model(model_name, config).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )
    loss_fn = nn.CrossEntropyLoss()

    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    best_miou = -1.0
    best_epoch = 0
    stale_epochs = 0
    best_confusion = None

    print(f"Training {model_name} with config: {config}")
    print(f"Best checkpoint target: {best_model_path}")

    for epoch in range(1, config["max_epochs"] + 1):
        train_loss, train_acc, train_miou, _ = run_epoch(
            model, train_loader, loss_fn, device, optimizer
        )
        val_loss, val_acc, val_miou, val_confusion = run_epoch(
            model, val_loader, loss_fn, device
        )
        scheduler.step(val_miou)

        print(
            f"epoch={epoch:02d} "
            f"train_loss={train_loss:.4f} train_mIoU={train_miou:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_mIoU={val_miou:.4f}"
        )

        if val_miou > best_miou:
            best_miou = val_miou
            best_epoch = epoch
            stale_epochs = 0
            best_confusion = val_confusion.clone()
            torch.save(
                {
                    "model_name": model_name,
                    "epoch": best_epoch,
                    "validation_miou": best_miou,
                    "num_classes": NUM_CLASSES,
                    "image_width": WIDTH,
                    "image_height": HEIGHT,
                    "config": config,
                    "state_dict": model.state_dict(),
                },
                best_model_path,
            )
            print(f"saved best checkpoint: {best_model_path.name}")
        else:
            stale_epochs += 1
            if stale_epochs >= config["early_stopping_patience"]:
                print(
                    "early stopping after "
                    f"{config['early_stopping_patience']} epochs without validation mIoU improvement"
                )
                break

    return model, best_epoch, best_miou, best_confusion, device, best_model_path, config


model, best_epoch, best_miou, best_confusion, device, best_model_path, used_config = train_selected_model(MODEL_NAME)
print(f"Best {MODEL_NAME} epoch: {best_epoch}")
print(f"Best validation mIoU: {best_miou:.4f}")
print(f"Used config: {used_config}")
print(f"Only saved file: {best_model_path}")


---
## 8. Optional Validation Confusion Matrix

This cell keeps the confusion-matrix diagnostic available without saving any extra artifacts. Leave `SHOW_CONFUSION_MATRIX = False` when only the best `.pt` file is needed.


In [ ]:
SHOW_CONFUSION_MATRIX = False


def load_categories(category_file):
    rows = Path(category_file).read_text(encoding="utf-8", errors="replace").splitlines()
    categories = ["background"]
    for row in rows[1:]:
        fields = row.strip().split(maxsplit=1)
        if len(fields) == 2:
            categories.append(fields[1])
    return categories


def show_confusion_matrix(confusion, categories, top_k=25):
    cm = confusion.numpy().astype(np.float64)
    support = cm.sum(axis=1)
    selected = np.argsort(support)[::-1][:top_k]
    selected = selected[support[selected] > 0]
    normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    detail = normalized[np.ix_(selected, selected)]
    labels = [categories[index] for index in selected]

    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(detail, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"{MODEL_NAME} validation confusion matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(np.arange(len(labels)), labels=labels, rotation=70, ha="right", fontsize=7)
    ax.set_yticks(np.arange(len(labels)), labels=labels, fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


if SHOW_CONFUSION_MATRIX and best_confusion is not None:
    show_confusion_matrix(best_confusion, load_categories(DATA_ROOT / "category.txt"))


---
## 9. Summary

- The notebook is now self-contained in the main `9444` folder.
- `MODEL_NAME` selects either ResNet50-FCN or VGG16-FCN for a single run.
- Checkpoint selection uses validation mIoU only.
- The only file written by the notebook is `Best_Model/<MODEL_NAME>_best_model.pt`.
